In [1]:
# 1. imports
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import matplotlib.pyplot as plt
import torch.optim as optim
import os
from PIL import Image
from torchvision import transforms
from sklearn.model_selection import train_test_split
# 2. device
# 3. dataset + dataloader
# 4. model
# 5. loss
# 6. optimizer
# 7. training loop
# 8. validation loop


In [28]:
# 2. device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


In [29]:
# 3. dataset + dataloader

labels_path = os.path.join(r"D:\Datasets\Kaggle\CIFAR10\cifar10", "labels.txt")
train_dir = os.path.join(r"D:\Datasets\Kaggle\CIFAR10\cifar10", "train")
test_dir = os.path.join(r"D:\Datasets\Kaggle\CIFAR10\cifar10", "test")

with open(labels_path,"r", encoding="utf-8") as f:
    labels = [line.strip() for line in f if line.strip()]
print(labels)

rows = []
for label in labels:
    folder = os.path.join(train_dir, label)

    if not os.path.isdir(folder):
        print("Missing folder:", folder)
        continue
    
    for filename in os.listdir(folder):
        full_path = os.path.join(folder,filename)

        if os.path.isfile(full_path):
            rows.append({"label" : label, "path" : full_path})

df = pd.DataFrame(rows)
print(df.head())
print("Total training images = ", len(df))

['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
      label                                               path
0  airplane  D:\Datasets\Kaggle\CIFAR10\cifar10\train\airpl...
1  airplane  D:\Datasets\Kaggle\CIFAR10\cifar10\train\airpl...
2  airplane  D:\Datasets\Kaggle\CIFAR10\cifar10\train\airpl...
3  airplane  D:\Datasets\Kaggle\CIFAR10\cifar10\train\airpl...
4  airplane  D:\Datasets\Kaggle\CIFAR10\cifar10\train\airpl...
Total training images =  50000


In [30]:
transform = transforms.Compose([
    transforms.Resize((32,32)),
      transforms.ToTensor(),
      transforms.Normalize((0.5,0.5,0.5),
                           (0.5,0.5,0.5))
      ])

#splitting train data to create some val data
train_df, val_df = train_test_split(df, test_size=0.1, random_state=42, stratify=df["label"])


In [31]:
class CIFARDataset(Dataset):
    def __init__(self, df, transform = None):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        #mapping the labels to integers
        self.classes = sorted(self.df["label"].unique())
        self.classes_to_idx = {c:i for i,c in enumerate(self.classes)}

    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        path = self.df.loc[idx, "path"]
        label_str = self.df.loc[idx, "label"]
        y = self.classes_to_idx[label_str]

        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)

        return img, y

train_dataset = CIFARDataset(train_df, transform=transform)
val_dataset = CIFARDataset(val_df, transform=transform )

train_loader = DataLoader(train_dataset, batch_size= 64, shuffle=True, num_workers=0, pin_memory=False)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False,num_workers=0)

In [32]:
#Model
class SimpleCNN(nn.Module):
    def __init__(self, num_classes = 10):
        super().__init__()

        self.layer1 = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        self.layer2 = nn.Sequential(
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        self.layer3 = nn.Sequential(
            nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        self.classifierlayer = nn.Sequential(
            nn.AdaptiveAvgPool2d((1,1)),
            nn.Flatten(),
            nn.Linear(128,256),
            nn.ReLU(),
            nn.Dropout(p=0.5),
            nn.Linear(256,num_classes)
        )

    def forward(self,x):

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.classifierlayer(x)
        return x


In [33]:
#Training loop
def train_1_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0
    correct = 0
    total = 0

    for x,y in loader:
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = criterion(logits,y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * x.size(0)

        preds = logits.argmax(dim=1)
        correct += (preds == y).sum().item()
        total += y.size(0)

    avg_loss = running_loss / total
    acc = correct / total
    return avg_loss, acc

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0
    correct = 0
    total = 0

    for x,y in loader:
        x = x.to(device)
        y = y.to(device)

        logits = model(x)
        loss = criterion(logits,y)
        running_loss += loss.item() * x.size(0)

        preds = logits.argmax(dim=1)
        correct += (preds == y).sum().item()
        total += y.size(0)

    avg_loss = running_loss / total
    acc = correct / total
    return avg_loss, acc


#main training script

num_classes = len(train_dataset.classes)
model = SimpleCNN(num_classes=num_classes).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr = 1e-3)

epochs = 5
for epoch in range (1, epochs + 1):
    train_loss, train_acc = train_1_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)
    print(f"Epoch {epoch}: train loss {train_loss:.4f}, acc {train_acc:.3f} | val loss {val_loss:.4f}, acc {val_acc:.3f}")


Epoch 1: train loss 1.7967, acc 0.314 | val loss 1.6288, acc 0.397
Epoch 2: train loss 1.4722, acc 0.454 | val loss 1.3460, acc 0.501
Epoch 3: train loss 1.2942, acc 0.526 | val loss 1.1751, acc 0.579
Epoch 4: train loss 1.1784, acc 0.572 | val loss 1.1547, acc 0.593
Epoch 5: train loss 1.0903, acc 0.608 | val loss 1.0253, acc 0.634
